Open VS Code settings (Ctrl+, / Cmd+,)      
Search for python.analysis.typeCheckingMode     
Set it to "basic" (good default) or "strict" (catches more, including missing TypedDict keys, more aggressively

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [2]:
class AgentState(TypedDict):
    number1: int
    operation: str
    number2: int

    number3: int
    operation2: str
    number4: int

    finalNumber: int
    finalNumber2: int

In [3]:

def adder(state: AgentState) -> AgentState:
    """This node adds the first 2 numbers"""

    state["finalNumber"] = state["number1"] + state["number2"]

    return state


def subtractor(state: AgentState) -> AgentState:
    """This node subtracts the first 2 numbers"""

    state["finalNumber"] = state["number1"] - state["number2"]

    return state


def adder2(state: AgentState) -> AgentState:
    """This node adds the second 2 numbers"""

    state["finalNumber2"] = state["number3"] + state["number4"]

    return state


def subtractor2(state: AgentState) -> AgentState:
    """This node subtracts the second 2 numbers"""

    state["finalNumber2"] = state["number3"] - state["number4"]

    return state

In [8]:

# Router 1

def decide_next_node(state: AgentState) -> str:
    """This node will select the next node of the graph"""
    
    print(state)

    if state["operation"] == "+":
        return "addition_operation"
    
    elif state["operation"] == "-":
        return "subtraction_operation"
    raise ValueError(f"Unknown operation: {state['operation']}") 

# Router 2

def decide_next_node2(state: AgentState) -> str:
    """This node will select the next node of the graph"""

    if state["operation2"] == "+":
        return "addition_operation2"  # edge
    
    elif state["operation2"] == "-":
        return "subtraction_operation2"

In [9]:
graph = StateGraph(AgentState)

graph.add_node("add_node", adder)
graph.add_node("subtract_node", subtractor)

graph.add_node("add_node2", adder2)
graph.add_node("subtract_node2", subtractor2)

graph.add_node("router", lambda state: state)
graph.add_node("router2", lambda state: state)


graph.add_edge(START, "router")

# First Conditional Edge
graph.add_conditional_edges(
    "router",
    decide_next_node, # passthrough function
    {
        # Edge Node
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node"

        # "addition_operation" -> "add_node"
        #       ↑                   ↑
        #   return value      destination node
    }
)


# Move to Router2
graph.add_edge("add_node", "router2")
graph.add_edge("subtract_node", "router2")


# Second Conditional Edge
graph.add_conditional_edges(
    "router2",
    decide_next_node2,
    {
        # Edge Node
        "addition_operation2": "add_node2",
        "subtraction_operation2": "subtract_node2"

        # "addition_operation2" -> "add_node2"
        #        ↑                    ↑
        #   return value      destination node
    }
)

# End
graph.add_edge("add_node2", END)
graph.add_edge("subtract_node2", END)

app = graph.compile()



In [10]:
# from IPython.display import Image, display
# display(Image(app.get_graph().draw_mermaid_png()))

In [13]:
initial_state = AgentState(
    number1=10,
    operation="-",
    number2=5,

    number3=7,
    operation2="+",
    number4=2

)

print(app.invoke(initial_state))

{'number1': 10, 'operation': '-', 'number2': 5, 'number3': 7, 'operation2': '+', 'number4': 2}
{'number1': 10, 'operation': '-', 'number2': 5, 'number3': 7, 'operation2': '+', 'number4': 2, 'finalNumber': 5, 'finalNumber2': 9}


Fair question — if nothing enforces it, why bother? Turns out `TypedDict` earns its keep in a few real ways, even without runtime validation.

**1. It defines the graph's state schema — LangGraph actually reads this**

This is the big one, and it's not just decorative. LangGraph looks at your `TypedDict`'s keys to know **what channels/fields exist in the state** and how nodes are allowed to update them. When you write:

```python
class AgentState(TypedDict):
    number1: int
    finalNumber: int
```

LangGraph uses this to build the state schema for the graph — it's *not* purely cosmetic, it's how LangGraph knows the shape of data flowing through the graph, what keys to expect for merging updates, etc. Without declaring some schema, LangGraph has no way to know what fields your state has at all.

**2. `Annotated` types can change actual runtime behavior**

This is where `TypedDict` annotations stop being "just hints" and start doing real work. For example:

```python
from typing import Annotated
import operator

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
```

Here, the `Annotated[list, operator.add]` tells LangGraph: "when a node returns a new value for `messages`, don't overwrite the old list — **append/reduce** using `operator.add`." That's a real, functional runtime behavior driven directly by your type annotation. So `TypedDict` isn't purely decorative in LangGraph — this reducer mechanism is core to how state updates accumulate (e.g., chat history growing turn by turn instead of getting overwritten).

**3. IDE autocomplete and static checking (the "free" benefits)**

Even without running mypy explicitly, most editors (VS Code + Pylance, PyCharm) read `TypedDict` live and give you:
- Autocomplete on `state["..."]` keys
- Red squiggles/warnings if you typo a key or use the wrong type
- Inline hover docs

This is exactly how you'd have caught your `finalNumebr` typo and the `state["finalNumber"] == "-"` bug *before* running the code, if you had the type-checking extension active and paying attention to its warnings.

**4. Documentation for humans**

It tells anyone reading your code exactly what fields the state carries and their types, without having to trace through every node function. That's valuable even if Python itself ignores it.

**So the honest tradeoff:**

| | `TypedDict` | `pydantic.BaseModel` |
|---|---|---|
| Defines LangGraph's schema | ✅ | ✅ |
| Reducer behavior (`Annotated`) | ✅ | ✅ |
| IDE autocomplete/static checks | ✅ | ✅ |
| Runtime validation (missing/wrong-type fields) | ❌ | ✅ |
| Performance overhead | none | slightly more (validation cost) |

`TypedDict` is LangGraph's lightweight default because it's fast and requires no extra dependency, but it trades away runtime safety. If you're debugging annoying `KeyError`s like you've hit twice now, switching that one class to `pydantic.BaseModel` is a very reasonable upgrade — you lose a little speed, gain real validation.